In [1]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from lifetimes import BetaGeoFitter, GammaGammaFitter
from lifetimes.utils import calibration_and_holdout_data, summary_data_from_transaction_data

RAW_UCI = r"C:\Users\reinh\Downloads\Projects\Online Retail II UCI/online_retail_II.csv"
RAW_TELCO = r"C:\Users\reinh\Downloads\Projects\Telecom Customer Churn\WA_Fn-UseC_-Telco-Customer-Churn.csv"
OUT_DIR = r"C:\Users\reinh\Downloads\Projects\Customer_Segmentation_Churn"

In [2]:
df = pd.read_csv(RAW_UCI, encoding="ISO-8859-1", parse_dates=["InvoiceDate"])

df = df.dropna(subset=["Customer ID"])
df = df[~df["Invoice"].astype(str).str.startswith("C")]
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

NON_PRODUCT = {"POST", "D", "M", "C2", "DOT", "BANK CHARGES", "AMAZONFEE", "ADJUST", "ADJUST2", "CRUK", "PADS", "S", "B"}
df = df[~df["StockCode"].astype(str).str.upper().isin(NON_PRODUCT)].copy()

df["Customer ID"] = df["Customer ID"].astype(int)
df["Revenue"] = df["Quantity"] * df["Price"]
df.to_csv(f"{OUT_DIR}/online_retail_clean.csv", index=False)
print(f"Clean rows: {len(df):,} | Customers: {df['Customer ID'].nunique():,}")

Clean rows: 802,644 | Customers: 5,852


In [3]:
snapshot = df["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm = pd.DataFrame({
    "Recency": (snapshot - df.groupby("Customer ID")["InvoiceDate"].max()).dt.days,
    "Frequency": df.groupby("Customer ID")["Invoice"].nunique(),
    "Monetary": df.groupby("Customer ID")["Revenue"].sum(),
})

X_rfm = StandardScaler().fit_transform(np.log1p(rfm))
rfm["Cluster"] = KMeans(n_clusters=4, random_state=42).fit_predict(X_rfm)

print(rfm.groupby("Cluster").mean().round(1))
rfm.to_csv(f"{OUT_DIR}/rfm_table.csv")

C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=23.
  warnings.warn(


         Recency  Frequency  Monetary
Cluster                              
0           28.3        3.0     847.6
1           27.8       19.2   10888.6
2          393.6        1.4     325.1
3          225.8        5.1    1973.2


In [4]:
END = df["InvoiceDate"].max()
CUTOFF = END - pd.Timedelta(days=90)
past = df[df["InvoiceDate"] <= CUTOFF]

# Build features from pre-cutoff transactions
orders = past.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()
feat = orders.groupby("Customer ID").agg(
    last=("InvoiceDate", "max"),
    first=("InvoiceDate", "min"),
    frequency=("Invoice", "count"),
    monetary=("Revenue", "sum"),
)
feat["recency"] = (CUTOFF - feat["last"]).dt.days
feat["tenure"] = (CUTOFF - feat["first"]).dt.days
feat["avg_order"] = feat["monetary"] / feat["frequency"]

# Label customers who spent £0 in the 90 days after cutoff as churned
future_spend = df[df["InvoiceDate"] > CUTOFF].groupby("Customer ID")["Revenue"].sum()
feat["churned"] = (future_spend.reindex(feat.index).fillna(0) == 0).astype(int)

# Assign named segments based on recency
X_seg = StandardScaler().fit_transform(np.log1p(feat[["recency", "frequency", "monetary"]]))
feat["segment"] = KMeans(n_clusters=4, random_state=42).fit_predict(X_seg)
seg_order = feat.groupby("segment")["recency"].median().sort_values().index
feat["segment_name"] = feat["segment"].map(dict(zip(seg_order, ["Champions", "Recent", "At Risk", "Lost"])))

# Train & evaluate Churn Models on a holdout split
features = ["recency", "frequency", "monetary", "tenure", "avg_order"]
X, y = feat[features], feat["churned"]
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

lr = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_tr, y_tr)
gb = GradientBoostingClassifier(random_state=42).fit(X_tr, y_tr)

print(f"Logistic Regression ROC-AUC: {roc_auc_score(y_te, lr.predict_proba(X_te)[:, 1]):.3f}")
print(f"Gradient Boosting ROC-AUC:   {roc_auc_score(y_te, gb.predict_proba(X_te)[:, 1]):.3f}")

# Score every customer out-of-fold so no customer is scored by a model trained on them
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
feat["churn_probability"] = cross_val_predict(GradientBoostingClassifier(random_state=42), X, y, cv=cv, method="predict_proba")[:, 1]
feat["at_risk_flag"] = (feat["churn_probability"] >= 0.5).astype(int)

C:\Users\reinh\anaconda3\Lib\site-packages\sklearn\cluster\_kmeans.py:1382: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=21.
  warnings.warn(


Logistic Regression ROC-AUC: 0.787
Gradient Boosting ROC-AUC:   0.796


In [5]:
tel = pd.read_csv(RAW_TELCO)
tel["TotalCharges"] = pd.to_numeric(tel["TotalCharges"], errors="coerce")
tel = tel.dropna(subset=["TotalCharges"])

y_t = (tel["Churn"] == "Yes").astype(int)
X_t = tel.drop(columns=["customerID", "Churn"])

num_cols = X_t.select_dtypes(include="number").columns
cat_cols = X_t.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
])

X_tr_t, X_te_t, y_tr_t, y_te_t = train_test_split(X_t, y_t, test_size=0.2, stratify=y_t, random_state=42)

gb_telco = make_pipeline(preprocessor, GradientBoostingClassifier(random_state=42)).fit(X_tr_t, y_tr_t)
probs_t = gb_telco.predict_proba(X_te_t)[:, 1]

print(f"Telco ROC-AUC: {roc_auc_score(y_te_t, probs_t):.3f}")
print(classification_report(y_te_t, (probs_t >= 0.5).astype(int)))

Telco ROC-AUC: 0.839
              precision    recall  f1-score   support

           0       0.84      0.89      0.87      1033
           1       0.64      0.53      0.58       374

    accuracy                           0.80      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.79      0.80      0.79      1407



In [6]:
orders["Revenue"] = orders["Revenue"].clip(upper=50000)

summary = summary_data_from_transaction_data(
    orders, "Customer ID", "InvoiceDate",
    monetary_value_col="Revenue", observation_period_end=CUTOFF, freq="W"
)
repeat = summary[summary["frequency"] > 0]

# Fit BG/NBD (purchase frequency) and Gamma-Gamma (order value) models
bgf = BetaGeoFitter().fit(summary["frequency"], summary["recency"], summary["T"])
ggf = GammaGammaFitter().fit(repeat["frequency"], repeat["monetary_value"])

summary.loc[summary["frequency"] == 0, "monetary_value"] = repeat["monetary_value"].mean()
expected_orders = bgf.conditional_expected_number_of_purchases_up_to_time(52, summary["frequency"], summary["recency"], summary["T"])
expected_value = ggf.conditional_expected_average_profit(summary["frequency"], summary["monetary_value"])

summary["CLV_12m"] = expected_orders * expected_value

# Combine with out-of-fold churn scores to calculate revenue at risk
final = feat.reset_index().merge(summary[["CLV_12m"]].reset_index(), on="Customer ID", how="left")
final["value_at_risk"] = final["churn_probability"] * final["CLV_12m"]
final.to_csv(f"{OUT_DIR}/customer_scores_final.csv", index=False)

print(final.groupby("segment_name")[["churn_probability", "CLV_12m", "value_at_risk"]].mean().round(2))

# Holdout check: test repeat order predictions against actual orders in the last 90 days
inv_all = df.groupby(["Customer ID", "Invoice", "InvoiceDate"], as_index=False)["Revenue"].sum()
ch = calibration_and_holdout_data(inv_all, "Customer ID", "InvoiceDate", calibration_period_end=CUTOFF, observation_period_end=END, freq="W")
bgf_cal = BetaGeoFitter().fit(ch["frequency_cal"], ch["recency_cal"], ch["T_cal"])
ch["pred_holdout"] = bgf_cal.predict(ch["duration_holdout"], ch["frequency_cal"], ch["recency_cal"], ch["T_cal"])

print(f"\nHoldout (90d) | Actual orders: {ch['frequency_holdout'].sum():,.0f} | Predicted: {ch['pred_holdout'].sum():,.0f} | Corr: {ch['frequency_holdout'].corr(ch['pred_holdout']):.3f}")

              churn_probability  CLV_12m  value_at_risk
segment_name                                           
At Risk                    0.53   854.38         384.42
Champions                  0.16  5388.36         537.72
Lost                       0.80   282.23         216.74
Recent                     0.44   967.95         384.62

Holdout (90d) | Actual orders: 4,630 | Predicted: 3,648 | Corr: 0.734
